# HW5: Image Captioning

This notebook consolidates the entire HW5 codebase into a single file so you
can train and test on **Google Colab** (or any Jupyter environment) without
managing a multi-file project.

**How to use on Colab:**
1. Upload `data.p` to your Google Drive.
2. Run the *Setup* cell to mount Drive and install dependencies.
3. Set `DATA_PATH` in the *Configuration* cell to point at your `data.p`.
4. Fill in every `TODO` section, then run the cells top-to-bottom.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "tqdm", "torch", "torchvision", "pillow"], check=True)

import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive mounted.")
else:
    print("Not on Colab. Skipping Drive mount.")


In [ ]:
import os, math, pickle, random, re, collections
from types import SimpleNamespace
from typing import Any, Callable, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as tv_models
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image


In [ ]:
# Change DATA_PATH to wherever you uploaded data.p
DATA_PATH    = "../data/data.p"
CHKPT_PATH   = "../checkpoints"
DECODER_TYPE = "rnn"       # "rnn" or "transformer"
EPOCHS       = 15
LR           = 5e-4
BATCH_SIZE   = 64
HIDDEN_SIZE  = 512
WINDOW_SIZE  = 20

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {DEVICE}")


---
## Part 0: Text Processing  (`process_text.py`)

In [ ]:
# ─── process_text.py ─────────────────────────────────────────────────────────
import collections
from typing import Counter


def pad_captions(captions: list, window_size: int) -> None:
    """
    Pads each caption in-place to exactly window_size + 1 tokens
    by appending '<pad>' tokens.

    After calling preprocess_captions every caption looks like:
        ['<start>', w1, w2, ..., wN, '<end>']
    This function ensures every caption is exactly window_size + 1 tokens long
    so they can be stacked into a rectangular numpy array.

    Args:
        captions    - list of token lists, modified in-place
        window_size - int; the maximum caption length (excluding padding)

    TODO: For each caption, append enough '<pad>' tokens so that the caption
          has exactly window_size + 1 tokens total.
    """
    # TODO: implement padding
    pass


def unk_captions(captions: list, word_count: Counter, minimum_frequency: int) -> None:
    """
    Replaces infrequent tokens with '<unk>' in-place.

    Any token whose frequency in word_count is <= minimum_frequency is
    considered rare and should be replaced with '<unk>'.

    Args:
        captions          - list of token lists, modified in-place
        word_count        - Counter mapping token -> frequency
        minimum_frequency - int; tokens with count <= this value are replaced

    TODO: For each token in each caption, check its frequency in word_count.
          If the frequency is <= minimum_frequency, replace it with '<unk>'.
    """
    # TODO: implement unking
    pass


def build_word_dictionary(train_captions: list, test_captions: list) -> dict:
    """
    Builds a word-to-index mapping from the training captions and converts
    both train and test captions from token lists to integer index lists in-place.

    Steps:
        1. Iterate over every token in train_captions in order. The first time
           a token is seen, assign it the next available integer index (from 0).
           Replace the token in the list with its index.
        2. Iterate over every token in test_captions. Replace each token with
           its index from word2idx; for unknown tokens, use the index of '<unk>'.

    Args:
        train_captions - list of token lists (modified in-place -> int lists)
        test_captions  - list of token lists (modified in-place -> int lists)

    Returns:
        word2idx - dict mapping str token -> int index

    TODO: Build word2idx by scanning train_captions, then convert test_captions.
    """
    word2idx  = {}
    vocab_size = 0

    # TODO: fill word2idx and convert train_captions to integer indices

    # TODO: convert test_captions to integer indices
    #       (use word2idx.get(word, word2idx['<unk>']) for unseen words)

    return word2idx


---
## Part 1: Data Preprocessing  (`preprocessing.py`)

Loads the Flickr 8k dataset, extracts ResNet-50 image features, and saves
everything as a pickle file (`data.p`).  **You only need to run
`create_pickle` once** — after that, use `load_data_from_pickle` every time.


In [ ]:
def preprocess_captions(captions: list, window_size: int) -> None:
    for i, caption in enumerate(captions):
        caption_nopunct = re.sub(r"[^a-zA-Z0-9]+", ' ', caption.lower())
        clean_words = [w for w in caption_nopunct.split() if len(w) > 1 and w.isalpha()]
        captions[i] = ['<start>'] + clean_words[:window_size - 1] + ['<end>']


def get_image_features(image_names: list, data_folder: str, vis_subset: int = 100):
    resnet = tv_models.resnet50(weights=tv_models.ResNet50_Weights.IMAGENET1K_V1)
    feature_extractor = torch.nn.Sequential(*list(resnet.children())[:-2])
    feature_extractor.eval()

    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    gap = torch.nn.AdaptiveAvgPool2d((1, 1))
    image_features, vis_images = [], []

    pbar = tqdm(image_names)
    with torch.no_grad():
        for i, name in enumerate(pbar):
            pbar.set_description(f"[{i+1}/{len(image_names)}] {name}")
            with Image.open(f"{data_folder}/Images/{name}") as img:
                img_rgb   = img.convert("RGB")
                vis_arr   = np.array(img_rgb.resize((224, 224)))
                img_t     = preprocess(img_rgb).unsqueeze(0)
            feat = gap(feature_extractor(img_t)).squeeze(-1).squeeze(-1)
            image_features.append(feat[0].numpy())
            if i < vis_subset:
                vis_images.append(vis_arr)
    return image_features, vis_images


def load_raw_data(data_folder: str) -> dict:
    """Build data.p from the raw Flickr 8k folder (images + captions.txt)."""
    with open(f"{data_folder}/captions.txt") as f:
        examples = f.read().splitlines()[1:]

    img2caps = {}
    for ex in examples:
        name, cap = ex.split(",", 1)
        img2caps.setdefault(name, []).append(cap)

    imgs = list(img2caps.keys())
    random.seed(0); random.shuffle(imgs)
    test_imgs, train_imgs = imgs[:1000], imgs[1000:]

    def all_caps(names):
        return [c for n in names for c in img2caps[n]]

    train_caps = all_caps(train_imgs)
    test_caps  = all_caps(test_imgs)
    preprocess_captions(train_caps, WINDOW_SIZE)
    preprocess_captions(test_caps,  WINDOW_SIZE)

    wc = collections.Counter(w for cap in train_caps for w in cap)
    unk_captions(train_caps, wc, 50)
    unk_captions(test_caps,  wc, 50)
    pad_captions(train_caps, WINDOW_SIZE)
    pad_captions(test_caps,  WINDOW_SIZE)
    word2idx = build_word_dictionary(train_caps, test_caps)

    print("Extracting train features…")
    tr_feats, tr_imgs_vis = get_image_features(train_imgs, data_folder)
    print("Extracting test features…")
    te_feats, te_imgs_vis  = get_image_features(test_imgs,  data_folder)

    return dict(
        train_captions=np.array(train_caps),
        test_captions=np.array(test_caps),
        train_image_features=np.array(tr_feats),
        test_image_features=np.array(te_feats),
        train_images=np.array(tr_imgs_vis),
        test_images=np.array(te_imgs_vis),
        word2idx=word2idx,
        idx2word={v: k for k, v in word2idx.items()},
    )


def create_pickle(data_folder: str) -> None:
    """Run once to build data.p from raw Flickr 8k files."""
    data = load_raw_data(data_folder)
    with open(f"{data_folder}/data.p", "wb") as f:
        pickle.dump(data, f)
    print(f"Saved to {data_folder}/data.p")



---
## Load Data & Train / Test


In [ ]:
# Load the preprocessed data
with open(DATA_PATH, "rb") as f:
    data_dict = pickle.load(f)

feat_prep = lambda x: np.repeat(np.array(x).reshape(-1, 2048), 5, axis=0)

train_captions  = torch.tensor(np.array(data_dict["train_captions"]),         dtype=torch.long)
test_captions   = torch.tensor(np.array(data_dict["test_captions"]),          dtype=torch.long)
train_img_feats = torch.tensor(feat_prep(data_dict["train_image_features"]),  dtype=torch.float32)
test_img_feats  = torch.tensor(feat_prep(data_dict["test_image_features"]),   dtype=torch.float32)
word2idx        = data_dict["word2idx"]
idx2word        = data_dict.get("idx2word", {v: k for k, v in word2idx.items()})

device = torch.device(DEVICE)
train_captions  = train_captions.to(device)
test_captions   = test_captions.to(device)
train_img_feats = train_img_feats.to(device)
test_img_feats  = test_img_feats.to(device)

print(f"Train captions:  {train_captions.shape}")
print(f"Test captions:   {test_captions.shape}")
print(f"Train img feats: {train_img_feats.shape}")
print(f"Test img feats:  {test_img_feats.shape}")
print(f"Vocab size:      {len(word2idx)}")


---
## Part 2: Decoders  (`decoder.py`)


In [ ]:
class RNNDecoder(nn.Module):

    def __init__(self, vocab_size: int, hidden_size: int, window_size: int,
                 dropout: float = 0.4) -> None:
        super().__init__()
        self.vocab_size  = vocab_size
        self.hidden_size = hidden_size
        self.window_size = window_size

        # TODO: Define your layers
        #   - An image embedding network (Linear(2048 -> hidden_size))
        #   - A word embedding layer (nn.Embedding)
        #   - A GRU layer (nn.GRU, batch_first=True)
        #   - A classifier head (Linear(hidden_size -> vocab_size))
        #   - Dropout layers

    def forward(self, encoded_images: torch.Tensor, captions: torch.Tensor) -> torch.Tensor:
        """
        :param encoded_images: [batch_size x 2048]
        :param captions:       [batch_size x window_size]
        :return: logits        [batch_size x window_size x vocab_size]

        1. Embed images and reshape for GRU initial hidden state  [1 x B x hidden_size]
        2. Embed captions (with dropout)
        3. Feed embedded captions + hidden state into GRU
        4. Pass GRU output through classifier
        """
        # TODO:

        raise NotImplementedError("RNNDecoder Not Implemented Yet")


class TransformerDecoder(nn.Module):

    def __init__(self, vocab_size: int, hidden_size: int, window_size: int,
                 dropout: float = 0.2) -> None:
        super().__init__()
        self.vocab_size  = vocab_size
        self.hidden_size = hidden_size
        self.window_size = window_size

        # TODO: Define your layers
        #   - An image projection network (Linear(2048 -> hidden_size))
        #   - A PositionalEncoding layer for caption tokens
        #   - One or more TransformerBlock decoder layers
        #   - A classifier head (Linear(hidden_size -> vocab_size))

    def forward(self, encoded_images: torch.Tensor, captions: torch.Tensor) -> torch.Tensor:
        """
        :param encoded_images: [batch_size x 2048]
        :param captions:       [batch_size x window_size]
        :return: logits        [batch_size x window_size x vocab_size]

        1. Project + reshape the image embedding as cross-attention context
        2. Embed caption tokens and apply positional encoding
        3. Pass through TransformerBlock(s)
        4. Apply classifier head
        """
        # TODO:

        raise NotImplementedError("TransformerDecoder Not Implemented Yet")


---
## Part 3: ImageCaptionModel  (`model.py`)


In [ ]:
# ─── model.py ────────────────────────────────────────────────────────────────

class ImageCaptionModel(nn.Module):

    def __init__(self, decoder: nn.Module) -> None:
        super().__init__()
        self.decoder = decoder

    def forward(self, encoded_images: torch.Tensor, captions: torch.Tensor) -> torch.Tensor:
        return self.decoder(encoded_images, captions)

    def compile(self, optimizer, loss: Callable, metrics: list) -> None:
        self.optimizer         = optimizer
        self.loss_function     = loss
        self.accuracy_function = metrics[0]

    def train_epoch(self, train_captions: torch.Tensor,
                    train_image_features: torch.Tensor,
                    padding_index: int, batch_size: int = 30) -> tuple:
        """
        TODO: Runs one epoch over all training examples.

        :param train_captions:       [N x (WINDOW_SIZE+1)]  — includes <start> and <end>
        :param train_image_features: [N x 2048]
        :param padding_index:        token id of <pad>
        :param batch_size:           int
        :return: (perplexity, loss, accuracy) on the training set

        Steps:
          1. Call super().train() to put model in training mode.
          2. Shuffle examples (use torch.randperm).
          3. Loop over mini-batches:
             a. Slice decoder_input  = captions[:, :-1]   (<start> … wN)
                        decoder_labels = captions[:, 1:]    (w1 … <end>)
             b. Build mask where labels != padding_index.
             c. Zero gradients, forward, compute loss, backward, clip grads, step.
             d. Track running loss / accuracy.
          4. Return (perplexity, avg_loss, avg_accuracy).
        """
        super().train()

        # HINT: shuffle the training data at the start of each epoch.

        ## TODO: implement the training loop
        raise NotImplementedError("train_epoch Not Implemented Yet")

    def test(self, test_captions: torch.Tensor, test_image_features: torch.Tensor,
             padding_index: int, batch_size: int = 30) -> tuple:
        """DO NOT CHANGE"""
        num_batches = int(len(test_captions) / batch_size)
        total_loss = total_seen = total_correct = 0

        self.eval()
        with torch.no_grad():
            for index, end in enumerate(range(batch_size, len(test_captions) + 1, batch_size)):
                start = end - batch_size
                batch_image_features = test_image_features[start:end, :]
                decoder_input  = test_captions[start:end, :-1]
                decoder_labels = test_captions[start:end,  1:]

                probs = self(batch_image_features, decoder_input)
                mask  = decoder_labels != padding_index
                num_predictions = mask.float().sum()
                loss     = self.loss_function(probs, decoder_labels, mask)
                accuracy = self.accuracy_function(probs, decoder_labels, mask)

                total_loss    += loss.item() * num_predictions.item()
                total_seen    += num_predictions.item()
                total_correct += num_predictions.item() * accuracy

                avg_loss = float(total_loss / total_seen)
                avg_acc  = float(total_correct / total_seen)
                avg_prp  = np.exp(avg_loss)
                print(f"\r[Valid {index+1}/{num_batches}]\t loss={avg_loss:.3f}\t acc: {avg_acc:.3f}\t perp: {avg_prp:.3f}", end='')

        print()
        return avg_prp, avg_acc


def accuracy_function(prbs: torch.Tensor, labels: torch.Tensor, mask: torch.Tensor) -> float:
    """DO NOT CHANGE"""
    predictions = torch.argmax(prbs, dim=-1)
    correct     = (predictions == labels) & mask
    return (correct.float().sum() / mask.float().sum()).item()


def loss_function(prbs: torch.Tensor, labels: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """DO NOT CHANGE"""
    B, T, V     = prbs.shape
    prbs_flat   = prbs.reshape(B * T, V)
    labels_flat = labels.reshape(B * T).long()
    mask_flat   = mask.reshape(B * T).float()
    loss_unred  = F.cross_entropy(prbs_flat, labels_flat, reduction='none')
    return (loss_unred * mask_flat).sum() / mask_flat.sum()


In [ ]:
class AttentionMatrix(nn.Module):

    def __init__(self, use_mask: bool = False) -> None:
        super().__init__()
        self.use_mask = use_mask

    def forward(self, K: torch.Tensor, Q: torch.Tensor) -> torch.Tensor:
        """
        STUDENT MUST WRITE:

        Computes attention weights given key and query matrices.

        :param K: [batch_size x window_size_keys   x embedding_size]
        :param Q: [batch_size x window_size_queries x embedding_size]
        :return:  attention matrix [batch_size x window_size_queries x window_size_keys]
        """
        window_size_queries = Q.size(1)
        window_size_keys    = K.size(1)
        embedding_size_keys = K.size(2)

        # TODO: Implement scaled dot-product attention.
        #       If self.use_mask is True, apply a causal (upper-triangular) mask.

        raise NotImplementedError("AttentionMatrix Not Implemented Yet")


class AttentionHead(nn.Module):

    def __init__(self, input_size: int, output_size: int, is_self_attention: bool) -> None:
        super().__init__()
        self.use_mask = is_self_attention

        # TODO: Initialise K, V, Q linear projections and an AttentionMatrix instance.

    def forward(self, inputs_for_keys: torch.Tensor,
                inputs_for_values: torch.Tensor,
                inputs_for_queries: torch.Tensor) -> torch.Tensor:
        """
        Runs a single attention head.

        :param inputs_for_keys:    [batch_size x KEY_WINDOW_SIZE   x input_size]
        :param inputs_for_values:  [batch_size x KEY_WINDOW_SIZE   x input_size]
        :param inputs_for_queries: [batch_size x QUERY_WINDOW_SIZE x input_size]
        :return:                   [batch_size x QUERY_WINDOW_SIZE x output_size]
        """
        # TODO:
        # 1) Project inputs into K, V, Q.
        # 2) Compute attention weights via AttentionMatrix.
        # 3) Return the weighted sum of V.

        raise NotImplementedError("AttentionHead Not Implemented Yet")


class MultiHeadedAttention(nn.Module):

    def __init__(self, emb_sz: int, use_mask: bool) -> None:
        super().__init__()

        # TODO: Create 3 AttentionHeads each with output size emb_sz // 3.
        #       Add a final nn.Linear(3 * (emb_sz // 3), emb_sz) combine layer.

    def forward(self, inputs_for_keys: torch.Tensor,
                inputs_for_values: torch.Tensor,
                inputs_for_queries: torch.Tensor) -> torch.Tensor:
        """
        Runs multi-headed attention (3 heads).

        :param inputs_for_keys:    [batch_size x KEY_WINDOW_SIZE   x emb_sz]
        :param inputs_for_values:  [batch_size x KEY_WINDOW_SIZE   x emb_sz]
        :param inputs_for_queries: [batch_size x QUERY_WINDOW_SIZE x emb_sz]
        :return:                   [batch_size x QUERY_WINDOW_SIZE x emb_sz]
        """
        # TODO: Run 3 heads, concatenate outputs, pass through combine layer.

        raise NotImplementedError("MultiHeadedAttention Not Implemented Yet")


class TransformerBlock(nn.Module):

    def __init__(self, emb_sz: int, multiheaded: bool = False, dropout: float = 0.2) -> None:
        super().__init__()

        # TODO:
        # 1) Masked self-attention layer   (use MultiHeadedAttention if multiheaded else AttentionHead)
        # 2) Unmasked cross-attention layer
        # 3) Three LayerNorm layers
        # 4) Feed-forward sublayer:  Linear(emb_sz -> 4*emb_sz) -> ReLU -> Dropout -> Linear(4*emb_sz -> emb_sz)
        # 5) Dropout layer

    def forward(self, inputs: torch.Tensor, context_sequence: torch.Tensor) -> torch.Tensor:
        """
        One Transformer decoder block.

        :param inputs:           [batch_size x INPUT_SEQ_LENGTH   x emb_sz]
        :param context_sequence: [batch_size x CONTEXT_SEQ_LENGTH x emb_sz]
        :return:                 [batch_size x INPUT_SEQ_LENGTH   x emb_sz]
        """
        # TODO:
        # 1) Masked self-attention  + residual + LayerNorm
        # 2) Cross-attention        + residual + LayerNorm
        # 3) Feed-forward           + residual + LayerNorm

        raise NotImplementedError("TransformerBlock Not Implemented Yet")


def positional_encoding(length: int, depth: int) -> torch.Tensor:
    """
    Generates a sinusoidal positional encoding matrix.

    :param length: number of positions (sequence length)
    :param depth:  embedding dimension (must be even)
    :return:       FloatTensor of shape [length x depth]

    Hint: use alternating sin/cos at different frequencies.
    See https://www.tensorflow.org/text/tutorials/transformer#the_embedding_and_positional_encoding_layer
    """
    # TODO: Implement sinusoidal positional encoding.
    return torch.zeros(length, depth)   # placeholder — replace this!


class PositionalEncoding(nn.Module):

    def __init__(self, vocab_size: int, embed_size: int, window_size: int,
                 dropout: float = 0.2) -> None:
        super().__init__()
        self.embed_size = embed_size
        self.embedding  = nn.Embedding(vocab_size, embed_size)
        self.dropout    = nn.Dropout(dropout)
        pos_enc = positional_encoding(length=window_size, depth=embed_size)
        self.register_buffer('pos_encoding', pos_enc[:window_size, :])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        :param x: integer tensor of token ids [batch_size x window_size]
        :return:  float tensor [batch_size x window_size x embed_size]

        Steps:
          1. Embed x with self.embedding.
          2. Scale embeddings by sqrt(embed_size).
          3. Add self.pos_encoding (broadcast over batch).
          4. Apply dropout.
        """
        # TODO:
        raise NotImplementedError("PositionalEncoding Not Implemented Yet")


---
## Part 5: Training & Evaluation Utilities  (`assignment.py`)


In [ ]:
def save_model(model, decoder_type: str, chkpt_path: str = CHKPT_PATH) -> None:
    save_dir = os.path.join(chkpt_path, decoder_type)
    os.makedirs(save_dir, exist_ok=True)
    torch.save({
        "model_state_dict": model.state_dict(),
        "decoder_type":     decoder_type,
        "vocab_size":       model.decoder.vocab_size,
        "hidden_size":      model.decoder.hidden_size,
        "window_size":      model.decoder.window_size,
    }, os.path.join(save_dir, "model.pt"))
    print(f"Saved → {save_dir}/model.pt")


def load_model(decoder_type: str, chkpt_path: str = CHKPT_PATH, device=None):
    device = device or DEVICE
    ckpt = torch.load(os.path.join(chkpt_path, decoder_type, "model.pt"), map_location=device)
    cls  = RNNDecoder if ckpt["decoder_type"] == "rnn" else TransformerDecoder
    dec  = cls(ckpt["vocab_size"], ckpt["hidden_size"], ckpt["window_size"])
    mdl  = ImageCaptionModel(dec)
    mdl.load_state_dict(ckpt["model_state_dict"])
    mdl.to(device)
    print(f"Loaded from {chkpt_path}/{decoder_type}/model.pt")
    return mdl


def plotter(history: dict, save_dir: str, model_type: str) -> None:
    epochs = range(1, len(history["train_perp"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle(f"{model_type.upper()} Training Curves", fontsize=13)

    ax1.plot(epochs, history["train_perp"], label="Train", marker="o", markersize=4)
    if history["val_perp"]:
        ax1.plot(epochs, history["val_perp"], label="Val", marker="s", markersize=4)
    ax1.set(xlabel="Epoch", ylabel="Perplexity", title="Perplexity")
    ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, history["train_acc"], label="Train", marker="o", markersize=4)
    if history["val_acc"]:
        ax2.plot(epochs, history["val_acc"], label="Val", marker="s", markersize=4)
    ax2.set(xlabel="Epoch", ylabel="Accuracy", title="Accuracy")
    ax2.legend(); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(save_dir, "training_curves.png")
    plt.savefig(path, dpi=120, bbox_inches="tight"); plt.close(fig)
    print(f"Curves saved → {path}")


def train_model(model, captions, img_feats, pad_idx, valid=None) -> None:
    best_perp = float("inf")
    history   = {"train_perp": [], "train_loss": [], "train_acc": [],
                 "val_perp":   [], "val_acc":    []}

    pbar = tqdm(range(EPOCHS))
    for epoch in pbar:
        try:
            tr_perp, tr_loss, tr_acc = model.train_epoch(
                captions, img_feats, pad_idx, batch_size=BATCH_SIZE
            )
        except KeyboardInterrupt:
            if epoch > 0: print("\nStopped early."); break
            raise
        history["train_perp"].append(float(tr_perp))
        history["train_loss"].append(float(tr_loss))
        history["train_acc"].append(float(tr_acc))

        if valid is not None:
            val_perp, val_acc = model.test(*valid, pad_idx, batch_size=BATCH_SIZE)
            history["val_perp"].append(float(val_perp))
            history["val_acc"].append(float(val_acc))
            pbar.set_postfix(tr_loss=f"{tr_loss:.3f}", tr_acc=f"{tr_acc:.3f}",
                             val_perp=f"{val_perp:.3f}", val_acc=f"{val_acc:.3f}")
            if val_perp < best_perp:
                best_perp = val_perp
                save_model(model, DECODER_TYPE)
                tqdm.write(f"  ** Best saved (val_perp={val_perp:.3f}) **")
        else:
            pbar.set_postfix(loss=f"{tr_loss:.3f}", acc=f"{tr_acc:.3f}", perp=f"{tr_perp:.3f}")
            if tr_perp < best_perp:
                best_perp = tr_perp
                save_model(model, DECODER_TYPE)

    save_dir = os.path.join(CHKPT_PATH, DECODER_TYPE)
    os.makedirs(save_dir, exist_ok=True)
    plotter(history, save_dir, DECODER_TYPE)


In [ ]:
decoder_class = {"rnn": RNNDecoder, "transformer": TransformerDecoder}[DECODER_TYPE]

decoder = decoder_class(
    vocab_size  = len(word2idx),
    hidden_size = HIDDEN_SIZE,
    window_size = WINDOW_SIZE,
)

model = ImageCaptionModel(decoder).to(device)

optimizer = optim.Adam(model.parameters(), lr=LR)
model.compile(
    optimizer = optimizer,
    loss      = loss_function,
    metrics   = [accuracy_function],
)

train_model(
    model,
    train_captions,
    train_img_feats,
    word2idx["<pad>"],
    valid=(test_captions, test_img_feats),
)


In [ ]:
perplexity, accuracy = model.test(
    test_captions, test_img_feats, word2idx["<pad>"], batch_size=BATCH_SIZE
)
print(f"\nTest Perplexity: {perplexity:.3f}")
print(f"Test Accuracy:   {accuracy:.3%}")
